# 10 - Crawl LangChain Documentation with Tavily

## Objective

Reproduce the **web-crawling and document-preparation portion** of the instructor's documentation-helper project, while keeping ingestion into Pinecone for notebook 11.

```text
LangChain documentation URL
        ↓
Tavily Crawl
        ↓
Crawled page results
        ↓
Validation and deduplication
        ↓
LangChain Document objects
        ↓
Save reusable crawl artifacts locally
```

## What this notebook includes from the instructor's logic

- Environment loading
- Corporate SSL handling
- Reusable logging helpers inspired by `logger.py`
- Tavily crawl configuration
- `max_depth=2`
- `extract_depth="advanced"`
- Conversion of Tavily results into LangChain `Document` objects
- Original page URL stored as `metadata["source"]`
- Crawl success and failure checks
- Quality inspection and pipeline summary

## Intentional separation from the instructor's single ingestion script

This notebook **does not yet**:

- Split pages into 4,000-character chunks
- Generate Gemini embeddings
- Upsert records into Pinecone

Those operations will be implemented in:

```text
11_documentation_ingestion_gemini_pinecone.ipynb
```

Separating the stages lets us inspect crawl quality before spending Gemini embedding quota or Pinecone write units.

> Tavily crawling consumes credits. Start with the small validation crawl before attempting a broader crawl.


## 1. Optional package installation

Run this cell only if an import fails. Restart the kernel after installation.

The official Tavily-LangChain integration is provided by `langchain-tavily`.


In [ ]:
# %pip install -U \
#     langchain-core \
#     langchain-tavily \
#     python-dotenv \
#     truststore


## 2. Check the active Python kernel

The Python executable should point to the `nlp_env` Conda environment.


In [1]:
import sys


print("Python executable:", sys.executable)
print("Python version:", sys.version)


Python executable: C:\Users\inkk1r04\AppData\Local\anaconda3\envs\nlp_env\python.exe
Python version: 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]


## 3. Configure SSL before importing external API clients

This follows the same concern handled by the instructor's `ingestion.py`, adapted to the Windows corporate environment that has worked in the previous notebooks.


In [2]:
import truststore


truststore.inject_into_ssl()


print("Windows trust store injected into Python SSL.")


Windows trust store injected into Python SSL.


## 4. Import the crawling and document components


In [3]:
import asyncio
import hashlib
import json
import os
import re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse, urlunparse

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_tavily import TavilyCrawl


print("Imports completed successfully.")


Imports completed successfully.


## 5. Notebook-friendly logging helpers

The instructor keeps terminal formatting in `logger.py`. The same logical helpers are included here so every pipeline phase reports clear status messages.


In [4]:
class Colors:
    """ANSI color codes for terminals that support colored output."""

    PURPLE = "\033[95m"
    CYAN = "\033[96m"
    DARKCYAN = "\033[36m"
    BLUE = "\033[94m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    RED = "\033[91m"
    BOLD = "\033[1m"
    END = "\033[0m"


def log_info(message, color=Colors.CYAN):
    print(f"{color}INFO: {message}{Colors.END}")


def log_success(message):
    print(f"{Colors.GREEN}SUCCESS: {message}{Colors.END}")


def log_warning(message):
    print(f"{Colors.YELLOW}WARNING: {message}{Colors.END}")


def log_error(message):
    print(f"{Colors.RED}ERROR: {message}{Colors.END}")


def log_header(message):
    print()
    print(Colors.BOLD + Colors.PURPLE + "=" * 72 + Colors.END)
    print(Colors.BOLD + Colors.PURPLE + message + Colors.END)
    print(Colors.BOLD + Colors.PURPLE + "=" * 72 + Colors.END)
    print()


log_success("Notebook logging helpers are ready.")


SUCCESS: Notebook logging helpers are ready.


## 6. Locate and load `.env`

The expected variables are:

```env
TAVILY_API_KEY=your-tavily-key
PINECONE_API_KEY=your-pinecone-key
GOOGLE_API_KEY=your-google-key
DOCUMENTATION_INDEX_NAME=langchain-docs-helper-gemini
```

Only `TAVILY_API_KEY` is used for crawling, but the other variables are checked to make sure the next ingestion stage is already configured.


In [5]:
CURRENT_DIR = Path.cwd().resolve()


env_candidates = [
    CURRENT_DIR / ".env",
    CURRENT_DIR.parent / "langchain-course" / ".env",
]


ENV_PATH = next(
    (
        candidate
        for candidate in env_candidates
        if candidate.exists()
    ),
    None,
)


if ENV_PATH is None:
    raise FileNotFoundError(
        "Could not find .env in the current notebook folder "
        "or the sibling langchain-course folder."
    )


load_dotenv(dotenv_path=ENV_PATH)


required_now = ["TAVILY_API_KEY"]
required_next = [
    "GOOGLE_API_KEY",
    "PINECONE_API_KEY",
    "DOCUMENTATION_INDEX_NAME",
]


missing_now = [
    variable_name
    for variable_name in required_now
    if not os.getenv(variable_name)
]


if missing_now:
    raise ValueError(
        "Missing variables required for crawling: "
        + ", ".join(missing_now)
    )


print("Environment file:", ENV_PATH)
print("TAVILY_API_KEY loaded:", bool(os.getenv("TAVILY_API_KEY")))

for variable_name in required_next:
    print(
        f"{variable_name} ready for notebook 11:",
        bool(os.getenv(variable_name)),
    )


Environment file: C:\Users\inkk1r04\OneDrive - Kellogg Company\Desktop\Work\0. DS Capability\Courses\Langchain\langchain-course\.env
TAVILY_API_KEY loaded: True
GOOGLE_API_KEY ready for notebook 11: True
PINECONE_API_KEY ready for notebook 11: True
DOCUMENTATION_INDEX_NAME ready for notebook 11: True


WindowsPath('C:/Users/inkk1r04/OneDrive - Kellogg Company/Desktop/Work/0. DS Capability/Courses/Langchain/langchain-course/.env')

## 7. Define crawl configuration

The instructor crawls:

```text
https://python.langchain.com/
```

The current public LangChain documentation is served from:

```text
https://docs.langchain.com/oss/python/langchain/overview
```

The notebook therefore uses the current documentation root by default while keeping the URL easy to change.

### Two crawl profiles

**Validation profile**

- Small page limit
- Lower credit usage
- Used to confirm networking, response structure, and content quality

**Instructor-aligned profile**

- `max_depth=2`
- `extract_depth="advanced"`
- Broader page limit

Do the validation crawl first.


In [6]:
START_URL = (
    "https://docs.langchain.com/oss/python/langchain/overview"
)

ALLOWED_DOMAIN = "docs.langchain.com"

# Keep crawl results in a local folder beside the notebooks.
ARTIFACT_DIR = CURRENT_DIR / "documentation_crawl_artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

RAW_RESULTS_PATH = ARTIFACT_DIR / "langchain_docs_tavily_raw.json"
DOCUMENTS_PATH = ARTIFACT_DIR / "langchain_docs_documents.jsonl"
SUMMARY_PATH = ARTIFACT_DIR / "langchain_docs_crawl_summary.json"

# Small first run to validate the pipeline and conserve Tavily credits.
VALIDATION_CRAWL_CONFIG = {
    "max_depth": 1,
    "max_breadth": 5,
    "limit": 10,
    "extract_depth": "advanced",
    "format": "markdown",
    "include_images": False,
    "allow_external": False,
}

# Closer to the instructor's max_depth=2 and advanced extraction.
# The page cap remains explicit to prevent an unexpectedly large crawl.
INSTRUCTOR_ALIGNED_CRAWL_CONFIG = {
    "max_depth": 2,
    "max_breadth": 20,
    "limit": 100,
    "extract_depth": "advanced",
    "format": "markdown",
    "include_images": False,
    "allow_external": False,
}


print("Start URL:", START_URL)
print("Allowed domain:", ALLOWED_DOMAIN)
print("Artifact directory:", ARTIFACT_DIR)
print("Validation profile:", VALIDATION_CRAWL_CONFIG)
print("Instructor-aligned profile:", INSTRUCTOR_ALIGNED_CRAWL_CONFIG)


Start URL: https://docs.langchain.com/oss/python/langchain/overview
Allowed domain: docs.langchain.com
Artifact directory: C:\Users\inkk1r04\OneDrive - Kellogg Company\Desktop\Work\0. DS Capability\Courses\Langchain\langchain-course-jupyter\documentation_crawl_artifacts
Validation profile: {'max_depth': 1, 'max_breadth': 5, 'limit': 10, 'extract_depth': 'advanced', 'format': 'markdown', 'include_images': False, 'allow_external': False}
Instructor-aligned profile: {'max_depth': 2, 'max_breadth': 20, 'limit': 100, 'extract_depth': 'advanced', 'format': 'markdown', 'include_images': False, 'allow_external': False}


## 8. Estimate and control crawl scope

A crawl's actual credit usage depends on the pages processed and Tavily's current pricing rules. The `limit` parameter is therefore treated as a hard safety control in this learning notebook.

Start with:

```text
limit = 10
```

After inspecting the result, increase the limit only if necessary.


In [7]:
def display_crawl_plan(config):
    """Print the important crawl controls without making an API call."""

    log_header("TAVILY CRAWL PLAN")
    log_info(f"Start URL: {START_URL}")
    log_info(f"Maximum depth: {config['max_depth']}")
    log_info(f"Maximum breadth per level: {config['max_breadth']}")
    log_info(f"Maximum total pages: {config['limit']}")
    log_info(f"Extraction depth: {config['extract_depth']}")
    log_info(f"Output format: {config['format']}")
    log_info(f"External links allowed: {config['allow_external']}")


display_crawl_plan(VALIDATION_CRAWL_CONFIG)



TAVILY CRAWL PLAN

INFO: Start URL: https://docs.langchain.com/oss/python/langchain/overview
INFO: Maximum depth: 1
INFO: Maximum breadth per level: 5
INFO: Maximum total pages: 10
INFO: Extraction depth: advanced
INFO: Output format: markdown
INFO: External links allowed: False


## 9. Create the Tavily crawler

Creating the tool does not itself make an API request.

The current Tavily integration accepts crawl controls such as maximum depth, breadth, total page limit, extraction depth, content format, and external-link policy.


In [8]:
def create_tavily_crawler(config):
    """Create a TavilyCrawl tool from a configuration dictionary."""

    return TavilyCrawl(
        max_depth=config["max_depth"],
        max_breadth=config["max_breadth"],
        limit=config["limit"],
        extract_depth=config["extract_depth"],
        format=config["format"],
        include_images=config["include_images"],
        allow_external=config["allow_external"],
    )


validation_crawler = create_tavily_crawler(
    VALIDATION_CRAWL_CONFIG
)


print("Crawler object:", type(validation_crawler).__name__)


Crawler object: TavilyCrawl


## 10. Run the small validation crawl

> **This cell calls Tavily and consumes crawl credits.**

It is intentionally limited to at most 10 pages. Run it once, then save and reuse the response.

If the documentation site blocks a path or Tavily returns no pages, do not immediately increase the crawl limit. Inspect the response and error fields first.


In [9]:
# Set to True only when ready to make the Tavily API request.
RUN_VALIDATION_CRAWL = False


if RUN_VALIDATION_CRAWL:
    log_header("VALIDATION CRAWL")
    log_info("Starting Tavily validation crawl...")

    crawl_response = validation_crawler.invoke({
        "url": START_URL,
    })

    log_success("Tavily validation crawl completed.")
else:
    crawl_response = None
    log_warning(
        "Validation crawl not executed. Set RUN_VALIDATION_CRAWL=True "
        "when ready to consume Tavily credits."
    )



VALIDATION CRAWL

INFO: Starting Tavily validation crawl...
SUCCESS: Tavily validation crawl completed.


## 11. Inspect the raw response structure

The expected response contains a base URL and a `results` list. Each successful result should include at least:

```text
url
raw_content
```

The exact response can also contain usage, timing, images, or failure information depending on the integration version.


In [44]:
if crawl_response is None:
    log_warning("No crawl response is available yet.")
else:
    print("Response Python type:", type(crawl_response))

    if isinstance(crawl_response, dict):
        print("Top-level keys:", list(crawl_response.keys()))
        print("Base URL:", crawl_response.get("base_url"))
        print(
            "Number of result items:",
            len(crawl_response.get("results", [])),
        )

        failed_results = crawl_response.get(
            "failed_results",
            [],
        )

        print("Failed result count:", len(failed_results))


Response Python type: <class 'dict'>
Top-level keys: ['base_url', 'results', 'response_time', 'request_id']
Base URL: https://docs.langchain.com/oss/python/langchain/overview
Number of result items: 41
Failed result count: 0


In [45]:
crawl_response['results'][1]['url']

'https://docs.langchain.com/oss/python/langchain/tools'

In [46]:
crawl_response['results'][1]['raw_content']

'## Documentation Index\n\nFetch the complete documentation index at: </llms.txt>\n\nUse this file to discover all available pages before exploring further.\n\nFall AMA Series: Live sessions on building, evaluating, deploying, and continously improving agents with LangSmith. [Register now →](https://events.langchain.com/fall-product-series/)\n\n![light logo](https://mintcdn.com/langchain-5e9cc07a/nQm-sjd_MByLhgeW/images/brand/langchain-docs-dark-blue.png?fit=max&auto=format&n=nQm-sjd_MByLhgeW&q=85&s=5babf1a1962208fd7eed942fa2432ecb)\n![dark logo](https://mintcdn.com/langchain-5e9cc07a/nQm-sjd_MByLhgeW/images/brand/langchain-docs-light-blue.png?fit=max&auto=format&n=nQm-sjd_MByLhgeW&q=85&s=0bcd2a1f2599ed228bcedf0f535b45b1)\n\nPython\n\n### Get started\n\n### Core components\n\n### Middleware\n\n### Frontend\n\n### Advanced usage\n\n### Agent development\n\n### Production\n\n## On this page\n\n# Tools\n\n## [\u200b](#create-tools) Create tools\n\n### [\u200b](#basic-tool-definition) Basi

## 12. Save the raw Tavily response immediately

Saving the raw response allows later notebook cells and notebook 11 to be rerun without repeating the web crawl.

The file may contain public documentation text and URLs. The API key is not written to disk.


In [47]:
if crawl_response is None:
    log_warning("Nothing to save because the crawl has not run.")
else:
    RAW_RESULTS_PATH.write_text(
        json.dumps(
            crawl_response,
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        encoding="utf-8",
    )

    log_success(
        f"Saved raw crawl response to: {RAW_RESULTS_PATH}"
    )


SUCCESS: Saved raw crawl response to: C:\Users\inkk1r04\OneDrive - Kellogg Company\Desktop\Work\0. DS Capability\Courses\Langchain\langchain-course-jupyter\documentation_crawl_artifacts\langchain_docs_tavily_raw.json


## 13. Reload a previously saved response

Use this cell after restarting Jupyter. It avoids another Tavily API call.


In [48]:
if crawl_response is None and RAW_RESULTS_PATH.exists():
    crawl_response = json.loads(
        RAW_RESULTS_PATH.read_text(
            encoding="utf-8"
        )
    )

    log_success(
        f"Reloaded saved crawl response from: {RAW_RESULTS_PATH}"
    )
elif crawl_response is None:
    log_warning(
        "No in-memory response and no saved raw response were found."
    )


## 14. Normalize URLs

Documentation crawls can return the same page through URLs that differ only by a fragment, trailing slash, or query string. Normalization reduces accidental duplicates.


In [49]:
def normalize_url(url):
    """
    Normalize a URL for deduplication and safe metadata storage.
     
    The function:
    1. Removes leading and trailing whitespace.
    2. Converts repeated slashes in the URL path to one slash.
    3. Removes a trailing slash unless the path is the root path.
    4. Converts the scheme and domain to lowercase.
    5. Removes parameters, query strings, and fragments.
    """

    parsed = urlparse(url.strip())

    normalized_path = re.sub(
        r"/{2,}",               # matches two or more consecutive forward slashes
        "/",                    # replaces them with one slash
        parsed.path or "/",
    )

    if normalized_path != "/":
        normalized_path = normalized_path.rstrip("/")    # Removes a trailing slash unless the path is the root path

    return urlunparse((
        parsed.scheme.lower(),            # Example: "HTTPS" becomes "https".
        parsed.netloc.lower(),            # Example: "Docs.LangChain.com" becomes "docs.langchain.com".
        normalized_path,
        "",
        "",
        "",
    ))


example_url = (
    "https://docs.langchain.com/oss/python/langchain/overview/?x=1#section"
)

print("Original:", example_url)
print("Normalized:", normalize_url(example_url))


Original: https://docs.langchain.com/oss/python/langchain/overview/?x=1#section
Normalized: https://docs.langchain.com/oss/python/langchain/overview


## 15. Validate and deduplicate crawl results

A result is accepted only when:

- URL is present
- URL belongs to `docs.langchain.com`
- Extracted content is non-empty
- Extracted content has a minimum useful length
- Normalized URL has not already been accepted

Rejected items are recorded with a reason for debugging.


In [50]:
MIN_CONTENT_CHARACTERS = 200


def prepare_crawl_results(response):
    """Validate, normalize, and deduplicate Tavily crawl results."""

    if not isinstance(response, dict):
        raise TypeError(
            "Expected the Tavily response to be a dictionary."
        )

    accepted = []
    rejected = []
    seen_urls = set()

    for result_number, item in enumerate(
        response.get("results", [])
    ):
        raw_url = str(item.get("url", "")).strip()
        raw_content = str(
            item.get("raw_content", "") or ""
        ).strip()

        if not raw_url:
            rejected.append({
                "result_number": result_number,
                "reason": "missing_url",
            })
            continue

        normalized_url = normalize_url(raw_url)
        parsed_url = urlparse(normalized_url)

        if parsed_url.netloc != ALLOWED_DOMAIN:
            rejected.append({
                "result_number": result_number,
                "url": normalized_url,
                "reason": "outside_allowed_domain",
            })
            continue

        if normalized_url in seen_urls:
            rejected.append({
                "result_number": result_number,
                "url": normalized_url,
                "reason": "duplicate_url",
            })
            continue

        if not raw_content:
            rejected.append({
                "result_number": result_number,
                "url": normalized_url,
                "reason": "empty_content",
            })
            continue

        if len(raw_content) < MIN_CONTENT_CHARACTERS:
            rejected.append({
                "result_number": result_number,
                "url": normalized_url,
                "reason": "content_too_short",
                "content_length": len(raw_content),
            })
            continue

        seen_urls.add(normalized_url)

        accepted.append({
            "url": normalized_url,
            "raw_content": raw_content,
            "content_length": len(raw_content),
        })

    return accepted, rejected


## 16. Run crawl-result validation

This cell makes no API calls.


In [51]:
if crawl_response is None:
    accepted_results = []
    rejected_results = []
    log_warning("No crawl response is available for validation.")
else:
    accepted_results, rejected_results = prepare_crawl_results(
        crawl_response
    )

    log_header("CRAWL VALIDATION RESULTS")
    log_success(
        f"Accepted pages: {len(accepted_results)}"
    )

    if rejected_results:
        rejection_counts = Counter(
            item["reason"]
            for item in rejected_results
        )

        log_warning(
            f"Rejected items: {len(rejected_results)}"
        )
        print("Rejection reasons:", dict(rejection_counts))
    else:
        log_info("No crawl items were rejected.")



CRAWL VALIDATION RESULTS

SUCCESS: Accepted pages: 37
Rejection reasons: {'outside_allowed_domain': 4}


In [52]:
rejected_results

[{'result_number': 37,
  'url': 'https://langchain.com/careers',
  'reason': 'outside_allowed_domain'},
 {'result_number': 38,
  'url': 'https://interrupt.langchain.com/',
  'reason': 'outside_allowed_domain'},
 {'result_number': 39,
  'url': 'https://smith.langchain.com/',
  'reason': 'outside_allowed_domain'},
 {'result_number': 40,
  'url': 'https://chat.langchain.com/',
  'reason': 'outside_allowed_domain'}]

## 17. Inspect accepted URLs before processing content

Review the URLs carefully. A documentation crawl can include landing pages, navigation pages, changelogs, conceptual guides, and API-reference pages.


In [53]:
for page_number, item in enumerate(
    accepted_results,
    start=1,
):
    print(
        f"{page_number:3d}. "
        f"{item['content_length']:7,d} characters | "
        f"{item['url']}"
    )


  1.   7,334 characters | https://docs.langchain.com/oss/python/langchain/overview
  2.  26,524 characters | https://docs.langchain.com/oss/python/langchain/tools
  3.   4,148 characters | https://docs.langchain.com/use-these-docs
  4.  26,161 characters | https://docs.langchain.com/oss/python/langchain/context-engineering
  5.   8,432 characters | https://docs.langchain.com/oss/python/langchain/multi-agent
  6.  27,136 characters | https://docs.langchain.com/oss/python/langchain/long-term-memory
  7.   3,636 characters | https://docs.langchain.com/oss/python/langchain/mcp
  8.   3,033 characters | https://docs.langchain.com/oss/python/langchain/studio
  9.  11,343 characters | https://docs.langchain.com/oss/python/langchain/guardrails
 10.   2,518 characters | https://docs.langchain.com/build-overview
 11.  34,761 characters | https://docs.langchain.com/oss/python/langchain/streaming
 12.   1,697 characters | https://docs.langchain.com/oss/python/langchain/philosophy
 13.  13,415 char

## 18. Inspect several extracted pages

This quality check helps detect:

- Navigation noise
- Empty pages
- Repeated headers and footers
- Cookie text
- Broken Markdown
- Incomplete extraction

No Gemini or Pinecone calls occur here.


In [54]:
PREVIEW_PAGES = 3
PREVIEW_CHARACTERS = 1800


for page_number, item in enumerate(
    accepted_results[:PREVIEW_PAGES],
    start=1,
):
    print("=" * 100)
    print("Page:", page_number)
    print("URL:", item["url"])
    print("Character count:", item["content_length"])
    print("Content preview:")
    print(item["raw_content"][:PREVIEW_CHARACTERS])
    print()


Page: 1
URL: https://docs.langchain.com/oss/python/langchain/overview
Character count: 7334
Content preview:
## Documentation Index

Fetch the complete documentation index at: </llms.txt>

Use this file to discover all available pages before exploring further.

Fall AMA Series: Live sessions on building, evaluating, deploying, and continously improving agents with LangSmith. [Register now →](https://events.langchain.com/fall-product-series/)

![light logo](https://mintcdn.com/langchain-5e9cc07a/nQm-sjd_MByLhgeW/images/brand/langchain-docs-dark-blue.png?fit=max&auto=format&n=nQm-sjd_MByLhgeW&q=85&s=5babf1a1962208fd7eed942fa2432ecb)
![dark logo](https://mintcdn.com/langchain-5e9cc07a/nQm-sjd_MByLhgeW/images/brand/langchain-docs-light-blue.png?fit=max&auto=format&n=nQm-sjd_MByLhgeW&q=85&s=0bcd2a1f2599ed228bcedf0f535b45b1)

Python

### Get started

### Core components

### Middleware

### Frontend

### Advanced usage

### Agent development

### Production

## On this page

# LangChain over

## 19. Convert accepted pages into LangChain `Document` objects

The instructor stores the page URL in:

```python
metadata={"source": page_url}
```

We preserve that field and add transparent crawl metadata useful for filtering, debugging, and citations.


In [55]:
crawl_timestamp = datetime.now(
    timezone.utc
).isoformat()


all_docs = []


for page_number, item in enumerate(
    accepted_results
):
    source_url = item["url"]

    document = Document(
        page_content=item["raw_content"],
        metadata={
            "source": source_url,
            "source_domain": urlparse(source_url).netloc,
            "source_path": urlparse(source_url).path,
            "crawl_page_number": page_number,
            "crawl_timestamp_utc": crawl_timestamp,
            "crawler": "tavily",
            "content_format": "markdown",
        },
    )

    all_docs.append(document)


log_success(
    f"Created {len(all_docs)} LangChain Document objects."
)


if all_docs:
    print("First document metadata:")
    print(all_docs[0].metadata)
    print("First document preview:")
    print(all_docs[0].page_content[:1000])


SUCCESS: Created 37 LangChain Document objects.
First document metadata:
{'source': 'https://docs.langchain.com/oss/python/langchain/overview', 'source_domain': 'docs.langchain.com', 'source_path': '/oss/python/langchain/overview', 'crawl_page_number': 0, 'crawl_timestamp_utc': '2026-09-28T11:24:53.940458+00:00', 'crawler': 'tavily', 'content_format': 'markdown'}
First document preview:
## Documentation Index

Fetch the complete documentation index at: </llms.txt>

Use this file to discover all available pages before exploring further.

Fall AMA Series: Live sessions on building, evaluating, deploying, and continously improving agents with LangSmith. [Register now →](https://events.langchain.com/fall-product-series/)

![light logo](https://mintcdn.com/langchain-5e9cc07a/nQm-sjd_MByLhgeW/images/brand/langchain-docs-dark-blue.png?fit=max&auto=format&n=nQm-sjd_MByLhgeW&q=85&s=5babf1a1962208fd7eed942fa2432ecb)
![dark logo](https://mintcdn.com/langchain-5e9cc07a/nQm-sjd_MByLhgeW/images/bran

## 20. Check content-level duplicates

Different URLs can occasionally expose identical page content. A content hash helps detect exact duplicates before embedding.


In [56]:
def content_hash(text):
    """Create a stable SHA-256 hash of normalized page text."""

    normalized_text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return hashlib.sha256(
        normalized_text.encode("utf-8")
    ).hexdigest()


hash_to_sources = {}


for document in all_docs:
    page_hash = content_hash(
        document.page_content
    )

    document.metadata["page_content_hash"] = page_hash

    hash_to_sources.setdefault(
        page_hash,
        [],
    ).append(document.metadata["source"])


duplicate_content_groups = {
    page_hash: sources
    for page_hash, sources in hash_to_sources.items()
    if len(sources) > 1
}


print(
    "Exact duplicate-content groups:",
    len(duplicate_content_groups),
)


for page_hash, sources in duplicate_content_groups.items():
    print("=" * 80)
    print("Content hash:", page_hash)
    print("Sources:")

    for source in sources:
        print("-", source)


Exact duplicate-content groups: 0


## 21. Remove exact content duplicates

When two URLs contain identical extracted text, retain the first page and record the alternate URL in metadata.


In [57]:
deduplicated_docs = []
first_document_by_hash = {}


for document in all_docs:
    page_hash = document.metadata[
        "page_content_hash"
    ]

    if page_hash not in first_document_by_hash:
        document.metadata["alternate_sources"] = []
        first_document_by_hash[page_hash] = document
        deduplicated_docs.append(document)
    else:
        first_document_by_hash[page_hash].metadata[
            "alternate_sources"
        ].append(document.metadata["source"])


print("Documents before content deduplication:", len(all_docs))
print("Documents after content deduplication:", len(deduplicated_docs))


Documents before content deduplication: 37
Documents after content deduplication: 37


## 22. Metadata privacy and quality checks

Only public documentation URLs and technical crawl metadata should be saved. Local paths, API keys, and organization-specific folder names must not appear in document metadata.


In [58]:
BLOCKED_METADATA_TERMS = [
    "onedrive",
    "c:\\users",
    "appdata",
    "tavily_api_key",
    "pinecone_api_key",
    "google_api_key",
]


metadata_issues = []


for document_number, document in enumerate(
    deduplicated_docs
):
    metadata_text = json.dumps(
        document.metadata,
        ensure_ascii=False,
    ).lower()

    detected_terms = [
        term
        for term in BLOCKED_METADATA_TERMS
        if term.lower() in metadata_text
    ]

    if detected_terms:
        metadata_issues.append({
            "document_number": document_number,
            "detected_terms": detected_terms,
        })

    source_url = document.metadata.get(
        "source",
        "",
    )

    if not source_url.startswith("https://"):
        metadata_issues.append({
            "document_number": document_number,
            "reason": "invalid_source_url",
            "source": source_url,
        })


if metadata_issues:
    raise ValueError(
        f"Metadata checks failed: {metadata_issues}"
    )


log_success(
    "Metadata privacy and quality checks passed."
)


SUCCESS: Metadata privacy and quality checks passed.


## 23. Calculate crawl statistics

These statistics help determine whether a broader crawl is necessary before proceeding to chunking and embeddings.


In [59]:
if deduplicated_docs:
    page_lengths = [
        len(document.page_content)
        for document in deduplicated_docs
    ]

    crawl_summary = {
        "start_url": START_URL,
        "allowed_domain": ALLOWED_DOMAIN,
        "crawl_timestamp_utc": crawl_timestamp,
        "raw_result_count": len(
            crawl_response.get("results", [])
        ),
        "accepted_url_count": len(accepted_results),
        "rejected_result_count": len(rejected_results),
        "deduplicated_document_count": len(deduplicated_docs),
        "total_characters": sum(page_lengths),
        "minimum_page_characters": min(page_lengths),
        "maximum_page_characters": max(page_lengths),
        "average_page_characters": round(
            sum(page_lengths) / len(page_lengths),
            1,
        ),
    }

    log_header("CRAWL SUMMARY")

    for key, value in crawl_summary.items():
        print(f"{key}: {value}")
else:
    crawl_summary = {}
    log_warning("No documents are available for statistics.")



CRAWL SUMMARY

start_url: https://docs.langchain.com/oss/python/langchain/overview
allowed_domain: docs.langchain.com
crawl_timestamp_utc: 2026-09-28T11:24:53.940458+00:00
raw_result_count: 41
accepted_url_count: 37
rejected_result_count: 4
deduplicated_document_count: 37
total_characters: 473727
minimum_page_characters: 1675
maximum_page_characters: 61537
average_page_characters: 12803.4


## 24. Save prepared LangChain documents as JSON Lines

Each line contains one page with:

```text
page_content
metadata
```

Notebook 11 will load this file, create overlapping chunks, generate Gemini embeddings, and upload them to Pinecone.


In [60]:
if not deduplicated_docs:
    log_warning("No prepared documents are available to save.")
else:
    with DOCUMENTS_PATH.open(
        "w",
        encoding="utf-8",
    ) as output_file:
        for document in deduplicated_docs:
            output_file.write(
                json.dumps(
                    {
                        "page_content": document.page_content,
                        "metadata": document.metadata,
                    },
                    ensure_ascii=False,
                )
                + "\n"
            )

    SUMMARY_PATH.write_text(
        json.dumps(
            crawl_summary,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    log_success(
        f"Saved prepared documents to: {DOCUMENTS_PATH}"
    )
    log_success(
        f"Saved crawl summary to: {SUMMARY_PATH}"
    )


SUCCESS: Saved prepared documents to: C:\Users\inkk1r04\OneDrive - Kellogg Company\Desktop\Work\0. DS Capability\Courses\Langchain\langchain-course-jupyter\documentation_crawl_artifacts\langchain_docs_documents.jsonl
SUCCESS: Saved crawl summary to: C:\Users\inkk1r04\OneDrive - Kellogg Company\Desktop\Work\0. DS Capability\Courses\Langchain\langchain-course-jupyter\documentation_crawl_artifacts\langchain_docs_crawl_summary.json


## 25. Verify the saved artifact

This verification simulates how notebook 11 will load the crawl output.


In [61]:
if DOCUMENTS_PATH.exists():
    saved_rows = []

    with DOCUMENTS_PATH.open(
        "r",
        encoding="utf-8",
    ) as input_file:
        for line in input_file:
            if line.strip():
                saved_rows.append(
                    json.loads(line)
                )

    print("Saved document rows:", len(saved_rows))

    if saved_rows:
        print("First saved metadata:")
        print(saved_rows[0]["metadata"])
        print("First saved content preview:")
        print(saved_rows[0]["page_content"][:800])
else:
    log_warning("The prepared-document artifact does not exist yet.")


Saved document rows: 37
First saved metadata:
{'source': 'https://docs.langchain.com/oss/python/langchain/overview', 'source_domain': 'docs.langchain.com', 'source_path': '/oss/python/langchain/overview', 'crawl_page_number': 0, 'crawl_timestamp_utc': '2026-09-28T11:24:53.940458+00:00', 'crawler': 'tavily', 'content_format': 'markdown', 'page_content_hash': 'b7721a4893bc5070440abdd6b25e93bdfe5de1c505e698f4dfa8587546a5ad28', 'alternate_sources': []}
First saved content preview:
## Documentation Index

Fetch the complete documentation index at: </llms.txt>

Use this file to discover all available pages before exploring further.

Fall AMA Series: Live sessions on building, evaluating, deploying, and continously improving agents with LangSmith. [Register now →](https://events.langchain.com/fall-product-series/)

![light logo](https://mintcdn.com/langchain-5e9cc07a/nQm-sjd_MByLhgeW/images/brand/langchain-docs-dark-blue.png?fit=max&auto=format&n=nQm-sjd_MByLhgeW&q=85&s=5babf1a1962208fd7eed94

# Optional broader crawl

Run this only after the validation crawl has succeeded and you have inspected Tavily usage.

The instructor uses `max_depth=2` with advanced extraction. This profile follows that core configuration and caps the crawl at 100 pages for safety.

> Running this cell consumes substantially more Tavily credits than the validation crawl and replaces the in-memory response. Save the result immediately afterward.


In [43]:
# RUN_INSTRUCTOR_ALIGNED_CRAWL = False


# if RUN_INSTRUCTOR_ALIGNED_CRAWL:
#     broader_crawler = create_tavily_crawler(
#         INSTRUCTOR_ALIGNED_CRAWL_CONFIG
#     )

#     log_header("INSTRUCTOR-ALIGNED DOCUMENTATION CRAWL")
#     log_info("Starting broader Tavily crawl...")

#     crawl_response = broader_crawler.invoke({
#         "url": START_URL,
#     })

#     RAW_RESULTS_PATH.write_text(
#         json.dumps(
#             crawl_response,
#             indent=2,
#             ensure_ascii=False,
#             default=str,
#         ),
#         encoding="utf-8",
#     )

#     log_success(
#         f"Broader crawl completed and saved to {RAW_RESULTS_PATH}."
#     )
#     log_warning(
#         "Rerun the validation, conversion, deduplication, "
#         "and save cells after this broader crawl."
#     )



INSTRUCTOR-ALIGNED DOCUMENTATION CRAWL

INFO: Starting broader Tavily crawl...
SUCCESS: Broader crawl completed and saved to C:\Users\inkk1r04\OneDrive - Kellogg Company\Desktop\Work\0. DS Capability\Courses\Langchain\langchain-course-jupyter\documentation_crawl_artifacts\langchain_docs_tavily_raw.json.


## 26. What this notebook produces

After a successful crawl and validation, the folder should contain:

```text
documentation_crawl_artifacts/
├── langchain_docs_tavily_raw.json
├── langchain_docs_documents.jsonl
└── langchain_docs_crawl_summary.json
```

## Handoff to notebook 11

Notebook 11 will:

```text
Load langchain_docs_documents.jsonl
        ↓
Recreate LangChain Document objects
        ↓
RecursiveCharacterTextSplitter
chunk_size = 4000
chunk_overlap = 200
        ↓
Add chunk metadata and deterministic IDs
        ↓
Gemini gemini-embedding-001
output dimensionality = 1536
        ↓
Pinecone index: langchain-docs-helper-gemini
namespace: langchain-docs
```

## Important operational rule

Do not rerun the Tavily crawl every time the notebook starts. Reload the saved raw response or prepared JSONL artifact unless the source documentation needs to be refreshed.
